# 🔴 NEXUS — AI CyberHack Operations Agent
**QLoRA Training Pipeline | Google Colab T4**

```
SETUP ORDER:
Cell 1 → Mount Drive
Cell 2 → Clone GitHub
Cell 3 → Copy dataset dari Drive
Cell 4 → Install deps
Cell 5 → Validate dataset
Cell 6 → GPU check
Cell 7 → TRAIN
Cell 8 → Monitor (TensorBoard)
Cell 9 → Test inference
```

In [ ]:
# ╔══════════════════════════════════╗
# ║  CELL 1 — Mount Google Drive    ║
# ╚══════════════════════════════════╝
from google.colab import drive
drive.mount('/content/drive')
print('✅ Drive mounted')

In [ ]:
# ╔══════════════════════════════════╗
# ║  CELL 2 — Clone GitHub repo     ║
# ╚══════════════════════════════════╝
import os

REPO_URL   = 'https://github.com/dnx123532/ai_cyberhack.git'
COLAB_ROOT = '/content/ai_cyberhack'

if not os.path.exists(COLAB_ROOT):
    !git clone {REPO_URL} {COLAB_ROOT}
else:
    !cd {COLAB_ROOT} && git pull

%cd {COLAB_ROOT}
!ls -la
print('✅ Repo ready')

In [ ]:
# ╔══════════════════════════════════════════════════════╗
# ║  CELL 3 — Copy dataset dari Drive → Colab SSD       ║
# ║  (training dari /content lebih cepat dari Drive)    ║
# ╚══════════════════════════════════════════════════════╝
import shutil, os

DRIVE = '/content/drive/MyDrive/nexus-agent'
LOCAL = '/content/ai_cyberhack'

# Buat folder jika belum ada
os.makedirs(f'{LOCAL}/data/jsonl', exist_ok=True)

# Copy dataset files
files_to_copy = [
    'data/jsonl/nexus_v2_sharegpt_train.jsonl',
    'data/jsonl/nexus_v2_sharegpt_val.jsonl',
]

for f in files_to_copy:
    src = f'{DRIVE}/{f}'
    dst = f'{LOCAL}/{f}'
    if os.path.exists(src):
        shutil.copy(src, dst)
        size = os.path.getsize(dst)/1024
        print(f'  ✓ {f.split("/")[-1]} ({size:.0f} KB)')
    else:
        # Fallback: file sudah ada di repo (di-push ke GitHub)
        if os.path.exists(dst):
            size = os.path.getsize(dst)/1024
            print(f'  ✓ {f.split("/")[-1]} from repo ({size:.0f} KB)')
        else:
            print(f'  ✗ {f} — not found in Drive or repo!')

print('\n✅ Dataset ready')

In [ ]:
# ╔══════════════════════════════════╗
# ║  CELL 4 — Install dependencies  ║
# ╚══════════════════════════════════╝
!pip install -q \
    transformers==4.46.0 \
    peft==0.13.0 \
    trl==0.12.0 \
    bitsandbytes==0.44.1 \
    datasets==3.1.0 \
    accelerate==1.1.0 \
    sentencepiece \
    hf_transfer \
    tensorboard

print('✅ Dependencies installed')

In [ ]:
# ╔══════════════════════════════════╗
# ║  CELL 5 — Validate dataset      ║
# ╚══════════════════════════════════╝
!python training/validation.py \
    --file data/jsonl/nexus_v2_sharegpt_train.jsonl \
    --format sharegpt --save

In [ ]:
# ╔══════════════════════════════════╗
# ║  CELL 6 — GPU check             ║
# ╚══════════════════════════════════╝
import torch, os

# T4 optimizations
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'max_split_size_mb:128'

if torch.cuda.is_available():
    gpu   = torch.cuda.get_device_name(0)
    vram  = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'  GPU  : {gpu}')
    print(f'  VRAM : {vram:.1f} GB')
    if vram < 14:
        print('  ⚠️  VRAM < 14GB — kurangi max_seq_length atau pakai 4-bit lebih agresif')
    else:
        print('  ✅ VRAM cukup untuk 7B QLoRA (OpenHermes-2.5-Mistral-7B)')
else:
    print('  ❌ No GPU — training akan sangat lambat!')

In [ ]:
# ╔══════════════════════════════════╗
# ║  CELL 7 — TRAIN (main)          ║
# ╚══════════════════════════════════╝
# Estimasi waktu T4 (dataset test awal — 58 train samples):
# - 58 samples x 3 epoch / (batch=1, grad_accum=8) = ~21 steps
# - Per step ~3-5 detik = ~2-5 menit total (ini smoke test, bukan run final)

!python training/train.py

In [ ]:
# ╔══════════════════════════════════╗
# ║  CELL 8 — TensorBoard monitor   ║
# ╚══════════════════════════════════╝
%load_ext tensorboard
%tensorboard --logdir /content/ai_cyberhack/logs/training

In [ ]:
# ╔══════════════════════════════════╗
# ║  CELL 9 — Test inference        ║
# ╚══════════════════════════════════╝
import torch
from peft import AutoPeftModelForCausalLM
from transformers import AutoTokenizer

MODEL_DIR = '/content/ai_cyberhack/models/lora_adapter'

tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR, trust_remote_code=True)
model = AutoPeftModelForCausalLM.from_pretrained(
    MODEL_DIR, torch_dtype=torch.float16, device_map='auto'
)

def ask_nexus(prompt: str, max_tokens: int = 300) -> str:
    chat = [
        {'role': 'system',  'content': 'Kamu adalah NEXUS, AI cybersecurity agent.'},
        {'role': 'user',    'content': prompt},
    ]
    text = tokenizer.apply_chat_template(chat, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors='pt').to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=max_tokens,
                             do_sample=True, temperature=0.7, top_p=0.9,
                             pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)

# Test
test_prompts = [
    'Apa langkah pertama untuk recon domain target.com?',
    'Tool apa yang paling efektif untuk subdomain enumeration?',
    'Jelaskan workflow Kerberoasting dari awal sampai akhir.',
]

for p in test_prompts:
    print(f'\n❓ {p}')
    print(f'🤖 {ask_nexus(p)}')
    print('─' * 60)

In [ ]:
# ╔══════════════════════════════════════════════════════╗
# ║  CELL 10 — Merge LoRA → full model (optional)       ║
# ╚══════════════════════════════════════════════════════╝
from peft import AutoPeftModelForCausalLM
from transformers import AutoTokenizer
import torch, shutil

ADAPTER_DIR = '/content/ai_cyberhack/models/lora_adapter'
MERGED_DIR  = '/content/ai_cyberhack/models/merged_model'
DRIVE_DST   = '/content/drive/MyDrive/nexus-agent/models/merged_model'

model = AutoPeftModelForCausalLM.from_pretrained(
    ADAPTER_DIR, torch_dtype=torch.float16, device_map='auto'
)
model = model.merge_and_unload()
tokenizer = AutoTokenizer.from_pretrained(ADAPTER_DIR)

model.save_pretrained(MERGED_DIR)
tokenizer.save_pretrained(MERGED_DIR)
print(f'✅ Merged model saved: {MERGED_DIR}')

# Backup ke Drive
shutil.copytree(MERGED_DIR, DRIVE_DST, dirs_exist_ok=True)
print(f'✅ Backed up to Drive: {DRIVE_DST}')